<a href="https://colab.research.google.com/github/Prasanna-Mahajan-2006/Deepfake-Detector/blob/main/BASELINE%20MODEL%20%2B%20REGULARIZATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub

In [ ]:
#importing necessary libraries

import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import Sequence
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
!pip install kagglehub -q

import kagglehub

# downloading the dataset


print("Downloading dataset via kagglehub...")
# This will download the dataset and save the exact folder location to the 'path' variable
path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")

print(f"Success! Dataset downloaded to: {path}")

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

# Peek inside the downloaded directory to confirm the folder name (usually 'Dataset')
print("Contents of the downloaded path:", os.listdir(path))


In [ ]:
# Extracting the datasets into variables
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
# 1. Configuration
BATCH_SIZE = 64
IMG_SIZE = (224, 224)
EPOCHS = 50
# Updated to point to your newly extracted 100k dataset
DATASET_DIR = '/content/my_100k_dataset'

# 2. Data Loading
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# Optimize data loading for GPU
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

# 3. Data Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# 4. Base Model (Native Keras MobileNetV3 Small)
base_model = keras.applications.MobileNetV3Small(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Freeze base layers

# 5. Build the Final Model
inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)

# Pass through the base model (MobileNetV3 handles its own pixel scaling internally!)
x = base_model(x)

# Convert 3D feature maps to 1D feature vectors
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)

# 6. Compilation
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

# 7. Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint("best_deepfake_model.keras", save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss')
]

# 8. Training
print(f"Starting training for {EPOCHS} epochs...")
history = model.fit(
    train_dataset,
    epochs=EPOCH,
    validation_data=val_dataset,
    callbacks=callbacks
)

print("Training complete! Model saved as 'best_deepfake_model.keras'")

In [ ]:
IMG_SIZE = (224, 224)
EPOCHS = 30

# 2. MODEL ARCHITECTURE (High Regularization)
print("Building the model architecture...")
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name="data_augmentation")

base_model = keras.applications.MobileNetV3Small(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Freeze the base model to preserve ImageNet features

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)

# Regularization to prevent overfitting
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

# Final classification layer with L2 Regularization
outputs = layers.Dense(
    1, activation='sigmoid',
    kernel_regularizer=regularizers.l2(0.01)
)(x)

#model = keras.Model(inputs, outputs)

# 3. COMPILATION
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

callbacks = [
    keras.callbacks.ModelCheckpoint("best_deepfake_model.keras", save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss')
]

# 4. TRAINING
print("\nStarting training... (Make sure your GPU is on!)")
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks
)

# 5. POST-TRAINING METRICS & CONFUSION MATRIX
print("\n--- Generating Final Evaluation Metrics on Test Set ---")

# Evaluate basic metrics on the unseen Test set
test_loss, test_acc, test_prec, test_rec = model.evaluate(test_ds)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

# IMPORTANT: To generate a Confusion Matrix, we must load the test data WITHOUT shuffling
# Otherwise, our predictions won't line up with the true labels!
print("Re-loading test data for Confusion Matrix (shuffle=False)...")
test_ds_unshuffled = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Test",
    image_size=IMG_SIZE,
    batch_size=64,
    shuffle=False # <--- This is the magic key for a correct confusion matrix
)

# Extract true labels
y_true = np.concatenate([y for x, y in test_ds_unshuffled], axis=0)

# Generate predictions
print("Generating predictions...")
y_pred_probs = model.predict(test_ds_unshuffled)
y_pred = (y_pred_probs > 0.5).astype("int32")

# Classification Report (Includes F1 Score)
print("\n--- Detailed Classification Report ---")
print(classification_report(y_true, y_pred, target_names=['Real (0)', 'Fake (1)']))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Deepfake Detection Performance on Unseen Test Data')
plt.show()